### Install Required Libraries

This cell ensures that all necessary Python packages are installed. It uses `pip` to install `langchain-groq`, `langchain-core`, and `python-dotenv`, which are essential for interacting with the Groq API and building LangChain applications.

In [ ]:
# Install Required Libraries
!pip install -q langchain-groq langchain-core python-dotenv

### API Key Configuration

This cell handles the secure loading and configuration of API keys. It imports `os` for environment variables and `google.colab.userdata` for securely accessing user secrets in Colab. It then sets up `GROQ_API_KEY`, `LANGCHAIN_API_KEY`, and enables `LANGCHAIN_TRACING_V2` and `LANGCHAIN_PROJECT` for monitoring and debugging with LangChain.

In [ ]:
# Setup API Keys
import os
from google.colab import userdata


os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "resume-screening"

print("Keys loaded")

### LangChain Pipeline Definition

This section defines the core logic of the resume screening application using LangChain. It involves:

1.  **Initializing ChatGroq:** Sets up the language model from Groq, specifically `llama-3.3-70b-versatile`, for generating responses.
2.  **Defining Extraction Prompt:** Creates a `ChatPromptTemplate` to instruct the LLM to extract professional skills, tools, and years of experience from a given resume text.
3.  **Defining Scoring Prompt:** Creates another `ChatPromptTemplate` to instruct the LLM to compare extracted details with a job description and output a JSON object containing a fit score, an explanation, and a list of missing skills.
4.  **Building the Pipeline:** Constructs a sequential chain that first extracts details from the resume, then uses these details along with the job description to generate a score, and finally parses the output into a JSON format.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# 1. Initialize the Groq LLM with a CURRENT model
# llama-3.3-70b-versatile is currently one of their best and most stable
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)

# 2. Define the Extraction Instruction
extract_prompt = ChatPromptTemplate.from_template("""
Extract professional skills, tools, and years of experience from this resume.
Resume: {resume_text}
Constraint: Do NOT assume skills not explicitly mentioned.
""")

# 3. Define the Scoring Instruction
score_prompt = ChatPromptTemplate.from_template("""
Compare the Extracted Details with the Job Description.
Job Description: {job_description}
Extracted Details: {extracted_details}

Output a JSON object with these exact keys:
"fit_score": (integer 0-100),
"explanation": (reasoning),
"missing_skills": (list)
""")

# 4. Build the Chain (The Pipeline)
pipeline = (
    {
        "extracted_details": extract_prompt | llm,
        "job_description": lambda x: x["job_description"]
    }
    | score_prompt
    | llm
    | JsonOutputParser()
)

print("Pipeline logic updated with the new model!")

### Candidate Evaluation and Testing

This cell demonstrates the usage of the defined LangChain pipeline. It:

1.  **Defines a Job Description:** Specifies the requirements for a 'Python Developer' role.
2.  **Prepares Candidate Resumes:** Sets up three distinct candidate profiles (Strong, Average, Weak) with varying skill sets and experience levels.
3.  **Executes the Pipeline:** Iterates through each candidate, invokes the `pipeline` with their resume text and the job description, and prints the `fit_score` and `explanation` generated by the LLM. Error handling is included to catch potential issues during pipeline execution.

In [ ]:
# Define your Job Description
jd = "Python Developer: Requires Python, SQL, and 3+ years experience in Backend development."

# Define your 3 test cases
candidates = [
    {"name": "Strong", "text": "Senior Backend Dev with 5 years experience. Expert in Python and SQL. Built scalable APIs."},
    {"name": "Average", "text": "Junior Dev with 1 year experience. Knows Python but very little SQL. Fast learner."},
    {"name": "Weak", "text": "Professional Yoga Instructor with 8 years experience in wellness and mindfulness."}
]

for person in candidates:
    print(f"--- Processing {person['name']} Candidate ---")
    try:
        response = pipeline.invoke({
            "resume_text": person['text'],
            "job_description": jd
        })
        print(f"Score: {response['fit_score']}")
        print(f"Reason: {response['explanation']}\n")
    except Exception as e:
        print(f"An error occurred: {e}")
